In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df=pd.read_csv("Student Social Media And Mental Health Impact.csv")
df


,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,18,Female,Other,High School,YouTube,Education,1.9,89,6.1,2.2,7.9,Low,7.9
4996,18,Female,Other,High School,YouTube,Education,6.6,216,2.3,1.8,6.0,Very High,4.7
4997,19,Female,India,Undergraduate,LinkedIn,Education,3.8,151,5.4,1.4,7.5,Medium,7.0
4998,18,Male,Other,High School,Instagram,Entertainment,3.8,119,4.0,1.5,6.4,Medium,6.7


In [2]:
print(df.columns)

Index(['Age', 'Gender', 'Country', 'Academic_Level', 'Most_Used_Platform',
       'Purpose_Of_Use', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
       'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night',
       'Stress_Level', 'Mental_Health_Score'],
      dtype='object')


In [3]:
top_country=df["Country"].value_counts().index[:10].to_list()
top_country

['Other',
 'India',
 'USA',
 'Canada',
 'Australia',
 'UK',
 'Germany',
 'Mexico',
 'Turkey',
 'France']

In [4]:
def group_countrys(Country):
  if Country in top_country:
    return Country
  else:
    return "other"

In [5]:
df["grouped_top_contry"]=df["Country"].apply(group_countrys)
df["grouped_top_contry"].value_counts()

,count
grouped_top_contry,
Other,1880
other,1352
India,389
USA,355
Canada,230
Australia,198
UK,185
Germany,136
Mexico,94


In [6]:
df

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score,grouped_top_contry
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8,Other
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6,Other
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0,Canada
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3,Other
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,18,Female,Other,High School,YouTube,Education,1.9,89,6.1,2.2,7.9,Low,7.9,Other
4996,18,Female,Other,High School,YouTube,Education,6.6,216,2.3,1.8,6.0,Very High,4.7,Other
4997,19,Female,India,Undergraduate,LinkedIn,Education,3.8,151,5.4,1.4,7.5,Medium,7.0,India
4998,18,Male,Other,High School,Instagram,Entertainment,3.8,119,4.0,1.5,6.4,Medium,6.7,Other


In [7]:
from sklearn.model_selection import train_test_split
skewed_col=["Study_Hours"]
other_numeric_cols=["Age","Avg_Daily_Usage_Hours","Daily_Unlocks","Physical_Activity_Hours","Sleep_Hours_Per_Night"]
ordinal_cols=["Stress_Level"]
normal_cols=["Gender","Academic_Level","Most_Used_Platform","Purpose_Of_Use","grouped_top_contry"]
feature_cols=skewed_col+other_numeric_cols+ordinal_cols+normal_cols
x=df[feature_cols]
y=df["Mental_Health_Score"]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.30,random_state=42)

In [8]:
from numpy import log1p
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer,StandardScaler,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
skewed_pipeline=Pipeline(steps=[
    ("skew",FunctionTransformer(log1p)),
    ("scale",StandardScaler())
])
other_pipline=Pipeline(steps=[
    ("scale",StandardScaler())
])
ordinal_pipeline=Pipeline(steps=[
    ("oridnal",OrdinalEncoder(categories=[["Low","Medium","High","Very High"]]))

])
normal_pipeline=Pipeline(steps=[
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])
column_transformer=ColumnTransformer(transformers=[
    ("skewed",skewed_pipeline,skewed_col),
    ("other",other_pipline,other_numeric_cols),
    ("ordinal",ordinal_pipeline,ordinal_cols),
    ("normal",normal_pipeline,normal_cols)])

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

liner_pipline=Pipeline(steps=([
    ("column",column_transformer),
    ("regression",LinearRegression())
]))
liner_pipline.fit(x_train,y_train)
x_pred_test=liner_pipline.predict(x_test)
y_pred_train=liner_pipline.predict(x_train)


r2_test=r2_score(y_test,x_pred_test)
mae_test=mean_absolute_error(y_test,x_pred_test)
mse_test=mean_squared_error(y_test,x_pred_test)
r2_train=r2_score(y_train,y_pred_train)
mae_train=mean_absolute_error(y_train,y_pred_train)
mse_train=mean_squared_error(y_train,y_pred_train)

print(f"r2_test:{r2_test}")
print(f"r2_train:{r2_train}")
print(f"mae_test:{mae_test}")



r2_test:0.7597338655764386
r2_train:0.7254266761392683
mae_test:0.5085876308254499


In [10]:
from sklearn.ensemble import RandomForestRegressor

rf_regressor=Pipeline(steps=([
    ("column",column_transformer),
    ("rf_regression",RandomForestRegressor())
]))
rf_regressor.fit(x_train,y_train)
x_rf_test=rf_regressor.predict(x_test)
y_rf_train=rf_regressor.predict(x_train)

r2_test_rf=r2_score(y_test,x_rf_test)
mae_test_rf=mean_absolute_error(y_test,x_rf_test)

r2_train_rf=r2_score(y_train,y_rf_train)
mae_train_rf=mean_absolute_error(y_train,y_rf_train)

print(f"r2_test:{r2_test_rf}")
print(f"r2_train:{r2_train_rf}")
print(f"mae_test:{mae_test_rf}")



r2_test:0.890294681282881
r2_train:0.9828036160485365
mae_test:0.3239409555555557


In [11]:
from sklearn.model_selection import RandomizedSearchCV

param_grid={
    "rf_regression__n_estimators":[100,200,300],
    "rf_regression__max_depth":[5,10,15],
    "rf_regression__min_samples_split":[2,5,10],
    "rf_regression__min_samples_leaf":[1,2,3]}
random_search=RandomizedSearchCV(estimator=rf_regressor,param_distributions=param_grid,
                                     n_iter=15,cv=5,scoring="r2",random_state=42,n_jobs=-1)
random_search.fit(x_train,y_train)


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('column',
                                              ColumnTransformer(transformers=[('skewed',
                                                                               Pipeline(steps=[('skew',
                                                                                                FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                                               ('scale',
                                                                                                StandardScaler())]),
                                                                               ['Study_Hours']),
                                                                              ('other',
                                                                               Pipeline(steps=[('scale',
                                                                                                StandardScaler())]),
                                                                               ['Age',
                                                                                'Avg_Daily_Usage_Hours',
                                                                                'Daily_Unlocks',
                                                                                'Physical_Activity_Hours',
                                                                                'Sleep_Ho...
                                                                                'Academic_Level',
                                                                                'Most_Used_Platform',
                                                                                'Purpose_Of_Use',
                                                                                'grouped_top_contry'])])),
                                             ('rf_regression',
                                              RandomForestRegressor())]),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'rf_regression__max_depth': [5, 10, 15],
                                        'rf_regression__min_samples_leaf': [1,
                                                                            2,
                                                                            3],
                                        'rf_regression__min_samples_split': [2,
                                                                             5,
                                                                             10],
                                        'rf_regression__n_estimators': [100,
                                                                        200,
                                                                        300]},
                   random_state=42, scoring='r2')

In [12]:
import joblib
joblib.dump(rf_regressor,"Mental_health_modal.pkl")
print("modal saved sucessfully")

modal saved sucessfully
